In [22]:
"""
cnn_model_single.py — Entrenamiento de CNN para UNA combinación de ventanas
=============================================================================

Cambia los parámetros en la sección CONFIGURACIÓN y ejecuta.
"""
import os
import sys
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Apuntar a la raíz del repositorio
os.chdir(r'C:\Users\eneko\neural-network-forecasting')
sys.path.insert(0, os.getcwd())

# Imports del repositorio
from config import RANDOM_SEED, MODELS_DIR, FIGURES_DIR
from src.data_pipeline import load_data, get_train_test
from src.evaluation import compute_mae, save_results, count_parameters
from src.plotting import plot_training_curves

np.random.seed(RANDOM_SEED)
import tensorflow as tf
tf.random.set_seed(RANDOM_SEED)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")


# ============================================================================
# ▶▶▶ CONFIGURACIÓN — CAMBIA AQUÍ LOS PARÁMETROS ◀◀◀
# ============================================================================

INPUT_WINDOW = 5          # Ventana de entrada: 5, 10, 30 o 90
OUTPUT_WINDOW = 90        # Ventana de salida: 1, 5, 30 o 90
FILTERS = 32              # Filtros primera capa Conv1D
KERNEL_SIZE = 3           # Tamaño del kernel primera capa Conv1D
PADDING = 'same'          # 'same' o 'valid'
USE_MAXPOOL = False       # True → usa MaxPooling1D después de la primera Conv1D
POOL_SIZE = 2             # MaxPooling pool_size (solo si USE_MAXPOOL = True)
FILTERS_2 = 64            # Filtros segunda capa Conv1D (poner 0 para no usarla)
KERNEL_SIZE_2 = 3         # Tamaño del kernel segunda capa Conv1D
FILTERS_3 = 128             # Filtros tercera capa Conv1D (poner 0 para no usarla)
KERNEL_SIZE_3 = 3         # Tamaño del kernel tercera capa Conv1D
USE_GAP = True            # True → GlobalAveragePooling1D; False → Flatten
DENSE_1 = 64              # Neuronas primera capa densa
DENSE_2 = 32              # Neuronas segunda capa densa (poner 0 para no usarla)
DENSE_3 = 0               # Neuronas tercera capa densa (poner 0 para no usarla)
DROPOUT_RATE_1 = 0.0      # Dropout después de DENSE_1 (poner 0 para desactivar)
DROPOUT_RATE_2 = 0.0      # Dropout después de DENSE_2 (poner 0 para desactivar)
DROPOUT_RATE_3 = 0.0      # Dropout después de DENSE_3 (poner 0 para desactivar)
L2_REG = 0.0001           # Regularización L2 en capas Conv1D (poner 0 para desactivar)
LEARNING_RATE = 0.00001
EPOCHS = 1000
BATCH_SIZE = 64
PATIENCE = 120
VALIDATION_SPLIT = 0.1
MODEL_NAME = (f"CNN_f{FILTERS}_k{KERNEL_SIZE}_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}"
              f"{'_MP' if USE_MAXPOOL else ''}"
              f"{'_GAP' if USE_GAP else ''}")


# ============================================================================
# CONSTRUCCIÓN DEL MODELO
# ============================================================================

def build_cnn(input_window, n_features=23):
    layers = [
        Conv1D(FILTERS, kernel_size=KERNEL_SIZE, activation='relu',
               input_shape=(input_window, n_features), padding=PADDING,
               kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None),
    ]

    if USE_MAXPOOL:
        layers.append(MaxPooling1D(pool_size=POOL_SIZE))

    if FILTERS_2 > 0:
        layers.append(Conv1D(FILTERS_2, kernel_size=KERNEL_SIZE_2, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))

    if FILTERS_3 > 0:
        layers.append(Conv1D(FILTERS_3, kernel_size=KERNEL_SIZE_3, activation='relu',
                             padding='same',
                             kernel_regularizer=l2(L2_REG) if L2_REG > 0 else None))

    layers.append(GlobalAveragePooling1D() if USE_GAP else Flatten())

    layers.append(Dense(DENSE_1, activation='relu'))
    if DROPOUT_RATE_1 > 0:
        layers.append(Dropout(DROPOUT_RATE_1))

    if DENSE_2 > 0:
        layers.append(Dense(DENSE_2, activation='relu'))
        if DROPOUT_RATE_2 > 0:
            layers.append(Dropout(DROPOUT_RATE_2))

    if DENSE_3 > 0:
        layers.append(Dense(DENSE_3, activation='relu'))
        if DROPOUT_RATE_3 > 0:
            layers.append(Dropout(DROPOUT_RATE_3))

    layers.append(Dense(n_features))

    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='mae')
    return model


# ============================================================================
# ENTRENAMIENTO
# ============================================================================

print("Cargando datos...")
returns = load_data()

X_train, X_test, y_train, y_test = get_train_test(returns, INPUT_WINDOW, OUTPUT_WINDOW)

model = build_cnn(INPUT_WINDOW, n_features=X_train.shape[2])
n_params = count_parameters(model)

print(f"\n{'='*60}")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Parámetros: {n_params:,}")
print(f"{'='*60}")
model.summary()

"""early_stop = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)"""

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    shuffle=True,
    #callbacks=[early_stop],
    verbose=1
)


# ============================================================================
# EVALUACIÓN
# ============================================================================

y_pred_train = model.predict(X_train, verbose=0)
y_pred_test = model.predict(X_test, verbose=0)

mae_train = compute_mae(y_train, y_pred_train)
mae_test = compute_mae(y_test, y_pred_test)
mae_val = min(history.history['val_loss']) if 'val_loss' in history.history else None

print(f"\n{'='*60}")
print(f"  Resultados {MODEL_NAME}:")
print(f"    MAE Train: {mae_train:.6f}")
if mae_val:
    print(f"    MAE Val:   {mae_val:.6f}")
print(f"    MAE Test:  {mae_test:.6f}")
print(f"{'='*60}")

save_results(
    model_name=MODEL_NAME,
    model_type='convolutional',
    input_window=INPUT_WINDOW,
    output_window=OUTPUT_WINDOW,
    mae_train=mae_train,
    mae_test=mae_test,
    n_params=n_params,
    mae_val=mae_val
)

plot_training_curves(history, MODEL_NAME, INPUT_WINDOW, OUTPUT_WINDOW, save=True)

model.save(os.path.join(MODELS_DIR, f"{MODEL_NAME}.keras"))
print(f"Modelo guardado en: {MODELS_DIR}{MODEL_NAME}.keras")

TensorFlow version: 2.21.0
Cargando datos...


[*********************100%***********************]  23 of 23 completed


Datos cargados: 16200 días, 23 activos
Rango: 1962-01-03 → 2026-05-15
Ventana entrada=5, salida=90
  X_train: (14495, 5, 23) | y_train: (14495, 23)
  X_test:  (1611, 5, 23)  | y_test:  (1611, 23)

  Modelo: CNN_f32_k3_in5_out90_GAP
  Parámetros: 44,247


Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_26 (Conv1D)              │ (None, 5, 32)          │         2,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_27 (Conv1D)              │ (None, 5, 64)          │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_28 (Conv1D)              │ (None, 5, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_13     │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 23)             │           759 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,247 (172.84 KB)

 Trainable params: 44,247 (172.84 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0167 - val_loss: 0.0159
Epoch 2/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0159 - val_loss: 0.0152
Epoch 3/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0152 - val_loss: 0.0145
Epoch 4/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0146 - val_loss: 0.0139
Epoch 5/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0139 - val_loss: 0.0133
Epoch 6/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0133 - val_loss: 0.0127
Epoch 7/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0128 - val_loss: 0.0121
Epoch 8/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0122 - val_loss: 0.0116
Epoch 9/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0117 - val_loss: 0.0111
Epoch 10/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0111 - val_loss: 0.0106
Epoch 11/1000
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0106 - val_loss: 0.0101
Epoch 12/1000
204/204 ━━━━━━━━

KeyboardInterrupt: 